In [134]:
from typing import Callable

import pandas as pd
import polars as pl
from pathlib import Path
from datetime import datetime, date

def getBatch(inputFunction : Callable) -> Callable:
    def innerWrap(inSeries : pl.Series):
        return pl.Series(map(inputFunction, inSeries))
    return innerWrap

@getBatch
def stripString(x: str | None) -> str | None:
    if x:
        return x.strip()
    return None


@getBatch
def parseDate(x: str) -> date:
    return datetime.strptime(x,"%m-%d-%Y").date()


@getBatch
def parseFloat(x:str) -> float:
    return float(x.replace(",",'')) 

@getBatch
def int2String(x:int | None) -> str | None:
    if x:
        return f"{x}"
    return None

@getBatch
def zeroPadded7(x:str | None) -> str | None:
    if x:
    # temp= 7 - len(x)
        return f"{x:0>7}"
    return None

loc_cols = ['GFPipeID', 'PipeName', 'GFLocID', 'GF_LocID_Tag', 'LocID', 'Loc', 'GF_LocName', 'LocName', 'GF_FacilityType', 'GF_FacilityTypeGroup', 'ReportedFacilityType', 'LocSegment', 'LocZone', 'State', 'County', 'InterconnectingEntity', 'UpdatedTime']

fact_cols =['GasMonth','GasDay','Dataset','GFLocID','LocName','DesignCapacity','OperatingCapacity','TotalScheduledQuantity','OperationallyAvailableCapacity','IT','FlowDirection','Timestamp',]

raw_OA_path = Path('.').absolute().parent / 'downloads/enbridge/OA_raw'

pipe_configs_path = Path('.').absolute().parent / 'src/artifacts/configFiles/PipeConfigs.parquet'

pipesDf = pl.read_parquet(pipe_configs_path)

In [135]:
pipe_configs_path.parent

PosixPath('/home/sai/Documents/Coding/GasFundies/GFPackage/src/artifacts/configFiles')

In [136]:
temp_list =[]
for i in raw_OA_path.iterdir():
    pipe_code = i.name.split('_',1)[0].strip()
    temp = pd.read_csv(i)
    if not temp.empty:
        temp_list.append(temp.assign(PipeCode=pipe_code))
    
df_OA : pd.DataFrame= pd.concat(temp_list,ignore_index=True)
df = pl.from_dataframe(df=df_OA.astype(str))

In [137]:
OA_cols_gold = [ 'Eff_Gas_Day', 'Loc', 'Loc_Name', 'Flow_Ind_Desc', 'IT', 'Total_Design_Capacity', 'Operating_Capacity', 'Total_Scheduled_Quantity', 'Operationally_Available_Capacity', 'PipeCode' ] 

OA_gold_map ={
    'Eff_Gas_Day': 'GasDay',
    # 'Cycle_Desc' : 'CycleDesc',
    # 'Loc':'LocID',
    'Loc_Name':'LocName',
    'Flow_Ind_Desc': 'FlowDirection',
    # 'IT':'IT',
    'Total_Design_Capacity':'DesignCapacity',
    'Operating_Capacity':'OperatingCapacity',
    'Total_Scheduled_Quantity':'TotalScheduledQuantity',
    'Operationally_Available_Capacity':'OperationallyAvailableCapacity',
 }

In [138]:
pipesDf.collect_schema()

Schema([('PartitionKey', String),
        ('RowKey', String),
        ('GFPipeID', String),
        ('MetaCode', String),
        ('NoNoticeCode', String),
        ('ParentPipe', String),
        ('PipeCode', String),
        ('PipeName', String),
        ('PointCapCode', String),
        ('SegmentCapCode', String)])

In [139]:
LocsDf = df.select(OA_cols_gold).rename(mapping=OA_gold_map).select(["Loc","LocName","PipeCode"]).join(pipesDf["PipeCode","GFPipeID","PipeName"], on="PipeCode").unique()

LocsDf=LocsDf.with_columns(
    pl.col("Loc").str.strip_chars(),
    pl.col("Loc").str.strip_chars().str.zfill(7).alias("LocID")
    # pl.col("Loc").str.strip_chars().map_batches(zeroPadded7,return_dtype=pl.String).alias("LocID"),
).with_columns(
    pl.concat_str([
        pl.col("GFPipeID"),
        pl.col("LocID")
    ]).alias("GFLocID"),
    pl.lit(datetime.now()).cast(pl.Datetime).alias("UpdatedTime")

)



# LocsDf.write_csv(pipe_configs_path.parent / "OALocMeta.csv")

In [140]:
LocsDf

Loc,LocName,PipeCode,GFPipeID,PipeName,LocID,GFLocID,UpdatedTime
str,str,str,str,str,str,str,datetime[μs]
"""73970""","""Farmersville Interconnect, Mo…","""TE""","""120""","""Texas Eastern Transmission, LP""","""0073970""","""1200073970""",2026-03-23 00:01:52.553308
"""73157""","""UGI CENTRAL - Penn Fuel - McCo…","""TE""","""120""","""Texas Eastern Transmission, LP""","""0073157""","""1200073157""",2026-03-23 00:01:52.553308
"""59122""","""UCG RURAL RETREAT""","""ET""","""105""","""East Tennessee Natural Gas, LL…","""0059122""","""1050059122""",2026-03-23 00:01:52.553308
"""59313""","""TROUSDALE COUNTY RECEIVING""","""ET""","""105""","""East Tennessee Natural Gas, LL…","""0059313""","""1050059313""",2026-03-23 00:01:52.553308
"""73868""","""Rice Energy-Ohio/Boltz Ridge R…","""TE""","""120""","""Texas Eastern Transmission, LP""","""0073868""","""1200073868""",2026-03-23 00:01:52.553308
…,…,…,…,…,…,…,…
"""71219""","""Columbia Gas of Pa. - York co.…","""TE""","""120""","""Texas Eastern Transmission, LP""","""0071219""","""1200071219""",2026-03-23 00:01:52.553308
"""59167""","""COOKEVILLE EAST""","""ET""","""105""","""East Tennessee Natural Gas, LL…","""0059167""","""1050059167""",2026-03-23 00:01:52.553308
"""70313""","""Columbia, KY""","""TE""","""120""","""Texas Eastern Transmission, LP""","""0070313""","""1200070313""",2026-03-23 00:01:52.553308


In [141]:
metaDf =pl.read_csv(pipe_configs_path.parent / "metaLocs.csv")

In [142]:
meta_map = {
    'Loc St Abbrev':'State',
    'Loc Cnty': 'County',
    'Loc Zone':'LocZone',
    'Loc Type Ind':'ReportedFacilityType',
    }

meta_cols_selected = [
    'Loc St Abbrev',
    'Loc Cnty',
    'Loc Zone',
    'Loc Type Ind',
    "GFLocID"
]

In [143]:
loc_cols

['GFPipeID',
 'PipeName',
 'GFLocID',
 'GF_LocID_Tag',
 'LocID',
 'Loc',
 'GF_LocName',
 'LocName',
 'GF_FacilityType',
 'GF_FacilityTypeGroup',
 'ReportedFacilityType',
 'LocSegment',
 'LocZone',
 'State',
 'County',
 'InterconnectingEntity',
 'UpdatedTime']

In [144]:
LocsDf =LocsDf.join(metaDf[meta_cols_selected],on="GFLocID", how="left").rename(mapping=meta_map)

In [149]:
LocsDf.with_columns(
    *[pl.lit(None).cast(pl.String).alias(col_name) for col_name in loc_cols if col_name not in LocsDf.columns]
).select(loc_cols).fill_null("")\
.write_csv(pipe_configs_path.parent / "OALocMeta.csv")

In [146]:
silverOA = df.join(pipesDf["PipeCode","GFPipeID"], on="PipeCode", how="left").with_columns(
    pl.concat_str(
        [
            pl.col("GFPipeID"),
            pl.col("Loc").str.zfill(7)

        ]
    ).alias("GFLocID")
).select(pl.exclude(["PipeCode","GFPipeID"])).unique()

In [147]:
goldOADf=df.select(OA_cols_gold).rename(OA_gold_map).join(pipesDf["GFPipeID","PipeCode"],on="PipeCode",how="left")\
.with_columns(

    pl.col("GasDay").str.to_date(format="%m-%d-%Y"),
    
    pl.lit(datetime.now()).cast(pl.Datetime).alias("Timestamp"),

    pl.lit("OA").alias("Dataset"),

    pl.col("Loc").str.strip_chars(),

    pl.col("DesignCapacity").str.replace_all(",",""),
    pl.col("OperatingCapacity").str.replace_all(",",""),
    pl.col("TotalScheduledQuantity").str.replace_all(",",""),
    pl.col("OperationallyAvailableCapacity").str.replace_all(",",""),

)\
.with_columns(
    pl.col("Loc").str.zfill(7).alias("LocID"),

    pl.col("DesignCapacity").str.to_decimal(scale=2),
    pl.col("OperatingCapacity").str.to_decimal(scale=2),
    pl.col("TotalScheduledQuantity").str.to_decimal(scale=2),
    pl.col("OperationallyAvailableCapacity").str.to_decimal(scale=2),
)\
.with_columns(
    pl.concat_str([
        pl.col("GFPipeID"),
        pl.col("LocID")
    ]).alias("GFLocID"),
    
    pl.concat_str(
        [
            pl.col("GasDay").dt.year().cast(pl.String),
            pl.col("GasDay").dt.month().cast(pl.String).str.zfill(2)
        ]
    ).str.to_integer().alias("GasMonth")
).select(fact_cols)

goldOADf


GasMonth,GasDay,Dataset,GFLocID,LocName,DesignCapacity,OperatingCapacity,TotalScheduledQuantity,OperationallyAvailableCapacity,IT,FlowDirection,Timestamp
i64,date,str,str,str,"decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]",str,str,datetime[μs]
202603,2026-03-09,"""OA""","""1030095001""","""TGP Glancy Fork""",230420.00,230420.00,58016.00,172404.00,"""N""","""Delivery""",2026-03-23 00:01:52.695886
202603,2026-03-09,"""OA""","""1030095002""","""Mark West - Langley Processing""",230420.00,230420.00,58016.00,172404.00,"""N""","""Receipt""",2026-03-23 00:01:52.695886
202603,2026-03-06,"""OA""","""1030095001""","""TGP Glancy Fork""",230420.00,230420.00,58016.00,172404.00,"""N""","""Delivery""",2026-03-23 00:01:52.695886
202603,2026-03-06,"""OA""","""1030095002""","""Mark West - Langley Processing""",230420.00,230420.00,58016.00,172404.00,"""N""","""Receipt""",2026-03-23 00:01:52.695886
202602,2026-02-28,"""OA""","""11500N1001""","""NEXUS/DTE MR04""",1526550.00,1526550.00,423616.00,1102934.00,"""N""","""Delivery""",2026-03-23 00:01:52.695886
…,…,…,…,…,…,…,…,…,…,…,…
202602,2026-02-26,"""OA""","""10600G0014""","""KINETICA DEEPWATER EXPRESS (99…",158521.00,158521.00,44034.00,114487.00,"""Y""","""Delivery""",2026-03-23 00:01:52.695886
202602,2026-02-26,"""OA""","""10600G0015""","""TRANSCO (992107)""",377192.00,438927.00,0.00,438927.00,"""N""","""Delivery""",2026-03-23 00:01:52.695886
202602,2026-02-26,"""OA""","""10600G0025""","""BALDPATE REDELIVERY""",8233.00,8578.00,0.00,8578.00,"""N""","""Receipt""",2026-03-23 00:01:52.695886


In [153]:
LocsList = []
for i in pipe_configs_path.parent.iterdir():
    if i.is_file() and i.name.endswith(".csv") and not i.name.startswith("meta"):
        LocsList.append(pd.read_csv(i))

In [156]:
pl.from_dataframe(pd.concat(LocsList,ignore_index=True)).with_columns(
    pl.lit("Enbridge").alias("PartitionKey"),
    pl.col("GFLocID").alias("RowKey"),

).write_csv(pipe_configs_path.parent / "AllLocsInitial.csv")